# EAGF Notebook 2: Statistical Analysis

This notebook performs comprehensive statistical tests on the final verified results:
- Load 10-seed paired runs (seeds 42-51)
- Compute mean ± std and 95% bootstrap confidence intervals (n=1000)
- Wilcoxon signed-rank test for Trust Index comparison
- Effect size (r) calculation
- Corrected privacy metric and TI_certified support

**Statistical methods:** Bootstrap resampling, Wilcoxon signed-rank test (non-parametric), effect size computation

**Paper reference:** Section 5.1.3 (Statistical Analysis Protocol)

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
!git clone https://github.com/aliakarma/eagf.git
%cd eagf
!pip install -r requirements.txt

In [ ]:
import sys, os, warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml
from scipy import stats

print('Core imports ready.')

In [ ]:
from pathlib import Path

# Setup PROJECT_ROOT
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'eagf' and (PROJECT_ROOT / 'eagf').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'eagf'
if PROJECT_ROOT.name != 'eagf':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

In [ ]:
# Verify PROJECT_ROOT setup
print('Environment Summary:')
print('=' * 50)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Configs exist: {Path(PROJECT_ROOT) / "configs" in Path(PROJECT_ROOT).iterdir()}')
print(f'Results dir: {Path(PROJECT_ROOT) / "results" / "baseline_aif360_dp"}')
print('Ready to load results.')

## 1. Load Pre-Computed Results (10-Seed Paired Evaluation)

In [ ]:
import json
from pathlib import Path

# Define seeds and directories
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
BASELINE_DIR = Path(PROJECT_ROOT) / "results" / "baseline"
EAGF_DIR = Path(PROJECT_ROOT) / "results" / "eagf"

if not BASELINE_DIR.exists():
    raise ValueError(f"Baseline results not found: {BASELINE_DIR}")

if not EAGF_DIR.exists():
    raise ValueError(f"EAGF results not found: {EAGF_DIR}")

print("Using FINAL results directory:")
print(BASELINE_DIR)
print(EAGF_DIR)

print('Loading Pre-Computed Results')
print('=' * 60)
print(f'Baseline dir: {BASELINE_DIR}')
print(f'EAGF dir:     {EAGF_DIR}')
print(f'Requested seeds: {SEEDS}')

# Find paired seeds
baseline_seeds = set()
eagf_seeds = set()

for seed_dir in BASELINE_DIR.glob('seed_*'):
    try:
        seed = int(seed_dir.name.replace('seed_', ''))
        results_file = seed_dir / 'results.json'
        if results_file.exists():
            baseline_seeds.add(seed)
    except:
        pass

for seed_dir in EAGF_DIR.glob('seed_*'):
    try:
        seed = int(seed_dir.name.replace('seed_', ''))
        results_file = seed_dir / 'results.json'
        if results_file.exists():
            eagf_seeds.add(seed)
    except:
        pass

paired_seeds = sorted(list(baseline_seeds & eagf_seeds & set(SEEDS)))

print(f'\nPaired seeds found: {paired_seeds}')
print(f'Total paired runs: {len(paired_seeds)}')

# Load results for both baseline and EAGF
baseline_results = {}
eagf_results = {}

for seed in paired_seeds:
    baseline_file = BASELINE_DIR / f'seed_{seed}' / 'results.json'
    if baseline_file.exists():
        with open(baseline_file) as f:
            baseline_results[seed] = json.load(f)
    
    eagf_file = EAGF_DIR / f'seed_{seed}' / 'results.json'
    if eagf_file.exists():
        with open(eagf_file) as f:
            eagf_results[seed] = json.load(f)

print(f'\nLoaded {len(baseline_results)} baseline runs')
print(f'Loaded {len(eagf_results)} EAGF runs')


## 2. Compute Bootstrap Confidence Intervals (95%, n=1000 resamples)

In [ ]:
# Bootstrap confidence interval function
def bootstrap_ci(data, n_resamples=1000, confidence=0.95):
    """Compute bootstrap CI for mean of data."""
    rng = np.random.RandomState(42)
    bootstrap_means = []
    
    for _ in range(n_resamples):
        resampled = rng.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(resampled))
    
    alpha = 1 - confidence
    ci_lower = np.percentile(bootstrap_means, alpha/2 * 100)
    ci_upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    
    return {
        'mean': np.mean(data),
        'std': np.std(data),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
    }

# Compute statistics for all metrics
metrics = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'accountability', 'trust_index', 'trust_index_certified']

baseline_stats = {}
eagf_stats = {}

for metric in metrics:
    baseline_vals = np.array([baseline_results[s][metric] for s in paired_seeds if metric in baseline_results[s]])
    eagf_vals = np.array([eagf_results[s][metric] for s in paired_seeds if metric in eagf_results[s]])
    
    if len(baseline_vals) > 0:
        baseline_stats[metric] = bootstrap_ci(baseline_vals, n_resamples=1000)
    else:
        baseline_stats[metric] = {'mean': 0, 'std': 0, 'ci_lower': 0, 'ci_upper': 0}
    
    if len(eagf_vals) > 0:
        eagf_stats[metric] = bootstrap_ci(eagf_vals, n_resamples=1000)
    else:
        eagf_stats[metric] = {'mean': 0, 'std': 0, 'ci_lower': 0, 'ci_upper': 0}

print('\n' + '=' * 90)
print('Bootstrap Confidence Intervals (95%, n=1000 resamples)')
print('=' * 90)

# Baseline
print('\nBASELINE (AIF360-DP):')
print('-' * 90)
for metric in metrics:
    stats_dict = baseline_stats[metric]
    print(f'  {metric:25s}: {stats_dict["mean"]:.4f} ± {stats_dict["std"]:.4f}  '
          f'CI95 [{stats_dict["ci_lower"]:.4f}, {stats_dict["ci_upper"]:.4f}]')

# EAGF
print('\nEAGF:')
print('-' * 90)
for metric in metrics:
    stats_dict = eagf_stats[metric]
    print(f'  {metric:25s}: {stats_dict["mean"]:.4f} ± {stats_dict["std"]:.4f}  '
          f'CI95 [{stats_dict["ci_lower"]:.4f}, {stats_dict["ci_upper"]:.4f}]')


## 3. Wilcoxon Signed-Rank Test (Non-Parametric TI Comparison)

In [ ]:
# Extract Trust Index values for paired comparison
ti_baseline = np.array([baseline_results[s]['trust_index'] for s in paired_seeds])
ti_eagf = np.array([eagf_results[s]['trust_index'] for s in paired_seeds])

# Wilcoxon signed-rank test (non-parametric paired test)
w_stat, w_pval = stats.wilcoxon(ti_eagf, ti_baseline, method='approx')

# Effect size (r = Z / sqrt(N))
# For Wilcoxon, z-score approximation
z_score = stats.norm.ppf(1 - w_pval/2)
effect_size_r = z_score / np.sqrt(len(paired_seeds))
if effect_size_r > 1:
    effect_size_r = 1.0  # Cap at 1

print('\n' + '=' * 90)
print('Wilcoxon Signed-Rank Test (TI: EAGF vs Baseline)')
print('=' * 90)

print(f'\nBASELINE Trust Index:')
print(f'  Values:       {ti_baseline}')
print(f'  Mean:         {np.mean(ti_baseline):.6f}')
print(f'  Std:          {np.std(ti_baseline):.6f}')
print(f'  95% CI:       [{baseline_stats["trust_index"]["ci_lower"]:.6f}, {baseline_stats["trust_index"]["ci_upper"]:.6f}]')

print(f'\nEAGF Trust Index:')
print(f'  Values:       {ti_eagf}')
print(f'  Mean:         {np.mean(ti_eagf):.6f}')
print(f'  Std:          {np.std(ti_eagf):.6f}')
print(f'  95% CI:       [{eagf_stats["trust_index"]["ci_lower"]:.6f}, {eagf_stats["trust_index"]["ci_upper"]:.6f}]')

print(f'\nPaired Differences (EAGF - Baseline):')
differences = ti_eagf - ti_baseline
print(f'  Differences:  {differences}')
print(f'  Mean diff:    {np.mean(differences):+.6f}')
print(f'  Std diff:     {np.std(differences):.6f}')

print(f'\nWilcoxon Signed-Rank Test:')
print(f'  Test statistic (W):     {w_stat:.4f}')
print(f'  p-value:                {w_pval:.6f}')
print(f'  Significance level:     α = 0.05')
result = '✓ SIGNIFICANT' if w_pval < 0.05 else '✗ NOT SIGNIFICANT'
print(f'  Result:                 {result} ({w_pval:.6f} {"<" if w_pval < 0.05 else ">"} 0.05)')

print(f'\nEffect Size:')
print(f'  z-score:                {z_score:.6f}')
print(f'  Effect size (r):        {effect_size_r:.6f}')
print(f'  Interpretation:         Large effect (r > 0.5)' if abs(effect_size_r) > 0.5 else 'Medium effect (0.3 < r < 0.5)' if abs(effect_size_r) > 0.3 else 'Small effect (r < 0.3)')

print(f'\nRelative TI Improvement:')
rel_improve = (np.mean(ti_eagf) - np.mean(ti_baseline)) / np.mean(ti_baseline) * 100
print(f'  {np.mean(ti_baseline):.6f} → {np.mean(ti_eagf):.6f}')
print(f'  Improvement: +{rel_improve:.2f}%')

print(f'\nSummary Statistics:')
print(f'  Number of paired seeds (n):  {len(paired_seeds)}')
print(f'  Seeds used:                  {paired_seeds}')


## 4. TI_certified Analysis (Governance Constraint)

In [ ]:
# Analyze TI_certified (governance constraint)
ti_cert_baseline = np.array([baseline_results[s].get('trust_index_certified', 0) for s in paired_seeds])
ti_cert_eagf = np.array([eagf_results[s].get('trust_index_certified', 0) for s in paired_seeds])

print('\n' + '=' * 90)
print('TI_certified Analysis (Governance Constraint)')
print('=' * 90)

print(f'\nTI_certified enforces minimum per-pillar thresholds:')
print(f'  Clarity (C):       ≥ 0.80')
print(f'  Recall Parity (RP): ≥ 0.95')
print(f'  Privacy (P):       ≥ 0.80')
print(f'  Accountability (A): ≥ 0.85')

print(f'\nBaseline TI_certified:')
print(f'  Values:  {ti_cert_baseline}')
print(f'  Mean:    {np.mean(ti_cert_baseline):.6f}')

print(f'\nEAGF TI_certified:')
print(f'  Values:  {ti_cert_eagf}')
print(f'  Mean:    {np.mean(ti_cert_eagf):.6f}')

print(f'\nInterpretation:')
if np.mean(ti_cert_baseline) == 0 and np.mean(ti_cert_eagf) == 0:
    print(f'  ✓ No model satisfies all per-pillar thresholds simultaneously')
    print(f'    This is expected—highlights governance trade-offs and need for multi-objective balancing')
else:
    print(f'  Baseline: {int(np.sum(ti_cert_baseline > 0))}/{len(ti_cert_baseline)} seeds certified')
    print(f'  EAGF:     {int(np.sum(ti_cert_eagf > 0))}/{len(ti_cert_eagf)} seeds certified')


## 5. All-Metric Comparison Visualization

In [ ]:
# Create comparison plot for all metrics with bootstrap CIs
display_metrics = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'accountability', 'trust_index']
metric_labels = ['Accuracy', 'Recall Parity\n(RP)', 'Clarity\n(C)', 'Privacy\n(P)', 'Accountability\n(A)', 'Trust Index\n(TI)']

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(display_metrics))
width = 0.35

# Baseline bars with error bars
baseline_means = [baseline_stats[m]['mean'] for m in display_metrics]
baseline_cis = [(baseline_stats[m]['mean'] - baseline_stats[m]['ci_lower'],
                 baseline_stats[m]['ci_upper'] - baseline_stats[m]['mean']) 
                for m in display_metrics]
baseline_yerr = list(zip(*baseline_cis))

# EAGF bars with error bars
eagf_means = [eagf_stats[m]['mean'] for m in display_metrics]
eagf_cis = [(eagf_stats[m]['mean'] - eagf_stats[m]['ci_lower'],
             eagf_stats[m]['ci_upper'] - eagf_stats[m]['mean']) 
            for m in display_metrics]
eagf_yerr = list(zip(*eagf_cis))

bars1 = ax.bar(x - width/2, baseline_means, width, yerr=baseline_yerr, 
               label='Baseline (AIF360-DP)', color='#FF6B6B', 
               capsize=5, alpha=0.85, edgecolor='white', linewidth=1)
bars2 = ax.bar(x + width/2, eagf_means, width, yerr=eagf_yerr,
               label='EAGF', color='#4ECDC4',
               capsize=5, alpha=0.85, edgecolor='white', linewidth=1)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.03,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.03,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Metrics', fontsize=11, fontweight='bold')
ax.set_ylabel('Score', fontsize=11, fontweight='bold')
ax.set_title('Metric Comparison with Bootstrap 95% CI (10-Seed Paired Evaluation)', 
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_ylim(0, 1.2)
ax.legend(loc='upper left', fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'figures'), exist_ok=True)
fig_path = os.path.join(PROJECT_ROOT, 'figures', 'notebook2_bootstrap_ci.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')


## 6. Summary: Statistical Test Results

In [ ]:
print('\n' + '=' * 90)
print('FINAL SUMMARY: Statistical Test Results')
print('=' * 90)

print('\n1. TRUST INDEX (TI) — PRIMARY OUTCOME')
print('-' * 90)
print(f'   Baseline TI:  {np.mean(ti_baseline):.6f} ± {np.std(ti_baseline):.6f}')
print(f'                 95% CI: [{baseline_stats["trust_index"]["ci_lower"]:.6f}, {baseline_stats["trust_index"]["ci_upper"]:.6f}]')

print(f'\n   EAGF TI:      {np.mean(ti_eagf):.6f} ± {np.std(ti_eagf):.6f}')
print(f'                 95% CI: [{eagf_stats["trust_index"]["ci_lower"]:.6f}, {eagf_stats["trust_index"]["ci_upper"]:.6f}]')

print(f'\n   Improvement:  +{(np.mean(ti_eagf) - np.mean(ti_baseline)):.6f} ({rel_improve:.2f}%)')

print(f'\n   Statistical Test (Wilcoxon Signed-Rank):')
print(f'     W-statistic:  {w_stat:.4f}')
print(f'     p-value:      {w_pval:.6f}  ← {"✓ SIGNIFICANT (p < 0.05)" if w_pval < 0.05 else "NOT SIGNIFICANT (p ≥ 0.05)"}')
print(f'     Effect size:  r = {effect_size_r:.6f}  ← {"✓ LARGE (r > 0.5)" if abs(effect_size_r) > 0.5 else "Medium (0.3 < r < 0.5)"}')

print(f'\n2. SECONDARY OUTCOMES')
print('-' * 90)
print(f'   Recall Parity (Fairness):')
print(f'     Baseline: {baseline_stats["recall_parity"]["mean"]:.6f} → EAGF: {eagf_stats["recall_parity"]["mean"]:.6f}')
print(f'     Improvement: +{(eagf_stats["recall_parity"]["mean"] - baseline_stats["recall_parity"]["mean"]):.6f}')

print(f'\n   Privacy (Corrected Formula):')
print(f'     Baseline: {baseline_stats["privacy"]["mean"]:.6f} → EAGF: {eagf_stats["privacy"]["mean"]:.6f}')
print(f'     Improvement: +{(eagf_stats["privacy"]["mean"] - baseline_stats["privacy"]["mean"]):.6f}')

print(f'\n   Clarity (Transparency):')
print(f'     Baseline: {baseline_stats["clarity"]["mean"]:.6f} → EAGF: {eagf_stats["clarity"]["mean"]:.6f}')
print(f'     Improvement: +{(eagf_stats["clarity"]["mean"] - baseline_stats["clarity"]["mean"]):.6f}')

print(f'\n   Accountability:')
print(f'     Baseline: {baseline_stats["accountability"]["mean"]:.6f} → EAGF: {eagf_stats["accountability"]["mean"]:.6f}')
print(f'     Improvement: +{(eagf_stats["accountability"]["mean"] - baseline_stats["accountability"]["mean"]):.6f}')

print(f'\n3. STUDY DESIGN')
print('-' * 90)
print(f'   Number of paired seeds (n):      {len(paired_seeds)}')
print(f'   Seeds:                           {paired_seeds}')
print(f'   Bootstrap resamples:             1,000')
print(f'   Confidence level:                95%')
print(f'   Statistical test:                Wilcoxon signed-rank (non-parametric)')

print('\n' + '=' * 90)
